In [ ]:
import json
import pandas as pd
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

from gitsource import GithubRepositoryDataReader, chunk_documents
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from openai import OpenAI

import minsearch
load_dotenv()
client = OpenAI()



In [7]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
documents = [file.parse() for file in reader.read()]

q1_docs = [doc for doc in documents if doc["filename"].startswith("01-agentic-rag/lessons/")][:3]

class Questions(BaseModel):
    questions: list[str]

data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

def llm_structured(client, instructions, user_prompt, response_model, model="gpt-5.4-mini"):
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]
    response = client.responses.parse(
        model=model,
        input=messages,
        text_format=response_model
    )
    return response.output_parsed, response.usage

total_tokens = 0
for doc in q1_docs:
    user_prompt = json.dumps({
        "filename": doc["filename"],
        "content": doc["content"][:5000]  
    })
    result, usage = llm_structured(
        client,
        data_gen_instructions,
        user_prompt,
        Questions
    )
    total_tokens += usage.input_tokens
    print(f"Page: {doc['filename']}")
    print(f"\tInput tokens: {usage.input_tokens}")
    print(f"\tQuestions: {result.questions}")

average_tokens = total_tokens / len(q1_docs)
print(f"\nQ1: Average input tokens: {average_tokens}")



Page: 01-agentic-rag/lessons/01-intro.md
	Input tokens: 1021
	Questions: ['What is RAG actually doing to fix an LLM’s lack of up-to-date knowledge and access to my data?', 'Why does the course start by building a RAG system in plain Python instead of using a framework right away?', 'How do LLMs work at a basic level, and why are they described as next-word predictors?', 'What are the main weaknesses of LLMs that this course says RAG can help with?', 'What will the two parts of this module cover, and what changes in the agentic version?']
Page: 01-agentic-rag/lessons/02-environment.md
	Input tokens: 1287
	Questions: ['What do I need installed before starting this module besides Python, and do I have to use OpenAI specifically?', 'How do I create the project from scratch, and what packages should I install for the course?', 'What’s the recommended way to store my API key so I don’t accidentally commit it to Git?', 'How do I check that my OpenAI setup works inside Jupyter, and what should

In [16]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

chunk_dicts = []
for chunk in chunks:
    chunk_dicts.append({
        "content": chunk["content"],
        "filename": chunk["filename"],
        "start": chunk.get("start", 0)
    })

text_index = minsearch.Index(text_fields=["content"])
text_index.fit(chunk_dicts)

def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)


df_gt = pd.read_csv("ground-truth.csv")
ground_truth = df_gt.to_dict(orient="records")

first_q = ground_truth[0]
text_results = text_search(first_q["question"])
print(f"Question: {first_q['question']}")
print(f"Q2: Text Search: {text_results[0]['filename']}")

Question: What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?
Q2: Text Search: 01-agentic-rag/lessons/03-rag.md


In [17]:
from embedder import Embedder

embedder = Embedder()
chunk_texts = [chunk["content"] for chunk in chunk_dicts]
chunk_vectors = embedder.encode_batch(chunk_texts)

vector_index = minsearch.VectorSearch(
    keyword_fields=["filename"],
    numeric_fields=["start"]
)

vector_documents = []
for i, chunk in enumerate(chunk_dicts):
    vector_documents.append({
        'vector': chunk_vectors[i],
        'filename': chunk['filename'],
        'start': chunk['start'],
        'content': chunk['content']
    })

vector_index.fit(chunk_vectors, vector_documents)

def vector_search(query, num_results=5):
    query_vector = embedder.encode(query)
    return vector_index.search(query_vector, num_results=num_results)

vector_results = vector_search(first_q["question"])
print(f"Question: {first_q['question']}")
print(f"Q3: Vector Search: {vector_results[0]['filename']}")

Question: What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?
Q3: Vector Search: 01-agentic-rag/lessons/01-intro.md


In [18]:
def compute_relevance(q, search_function, num_results=5):
    """Compute relevance list for a single query."""
    filename = q["filename"]
    results = search_function(q["question"], num_results=num_results)
    return [1 if r.get("filename") == filename else 0 for r in results]

def hit_rate(relevance_total):
    if not relevance_total:
        return 0.0
    hits = sum(1 for rel in relevance_total if 1 in rel)
    return hits / len(relevance_total)

def mrr(relevance_total):
    if not relevance_total:
        return 0.0
    total = 0.0
    for rel in relevance_total:
        for rank, score in enumerate(rel):
            if score == 1:
                total += 1.0 / (rank + 1)
                break
    return total / len(relevance_total)

def evaluate(ground_truth, search_function, num_results=5):
    relevance_total = []
    for q in tqdm(ground_truth, desc="Evaluating"):
        relevance_total.append(compute_relevance(q, search_function, num_results))
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total)
    }

text_metrics = evaluate(ground_truth, text_search)
print(f"Q4: Text Search Hit Rate Answer: {text_metrics['hit_rate']:.2f}")

Evaluating:   0%|          | 0/360 [00:00<?, ?it/s]

Q4: Text Search Hit Rate Answer: 0.76


In [19]:
vector_metrics = evaluate(ground_truth, vector_search)
print(f"Q5: Vector Search MRR: {vector_metrics['mrr']:.2f}")

Evaluating:   0%|          | 0/360 [00:00<?, ?it/s]

Q5: Vector Search MRR: 0.55


In [21]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc.get("start", 0))
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            if key not in docs:
                docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60, num_results=5):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k, num_results=num_results)

k_values = [1, 50, 100, 200]
results = {}

for k in k_values:
    def search_with_k(query, num_results=5):
        return hybrid_search(query, k=k, num_results=num_results)
    
    metrics = evaluate(ground_truth, search_with_k)
    results[k] = metrics

best_k = min(k_values, key=lambda k: (-results[k]['mrr'], k))
print(f"\nBest k by MRR: {best_k}")

Evaluating:   0%|          | 0/360 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/360 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/360 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/360 [00:00<?, ?it/s]


Best k by MRR: 1
